# Pure Weber Planetary Atom: Elliptical Orbit — Live ($\eta = 0.8$, $a = 0$)

This notebook demonstrates a **bound three-body state** in Weber
electrodynamics: two like charges forming a sub-critical nucleus, orbited
by an unlike charge in a wide elliptical orbit.

## Physics Background

### Weber's Velocity-Dependent Force

Weber's force law between two charges $q_i$, $q_j$ separated by distance
$r$ with radial velocity $\dot{r}$ and acceleration $\ddot{r}$ is

$$F = \frac{q_i q_j}{r^2}\left(1 - \frac{\dot{r}^2}{2c^2} + \frac{r\ddot{r}}{c^2}\right)$$

The velocity- and acceleration-dependent terms create an **effective
inertial mass** that depends on separation:

$$\mu_{\text{eff}}(r) = \mu\left(1 - \frac{\rho}{r}\right)$$

where $\mu$ is the reduced mass and $\rho = q_i q_j / (\mu c^2)$ is
the **critical radius**.

### Critical Radius and Sub-Critical Binding

For two like charges ($q_i q_j > 0$), the critical radius $\rho > 0$ is
a real positive distance. The effective inertial mass changes sign at
$r = \rho$, creating two permanently separated dynamical regimes:

- **Distant state** ($r > \rho$): ordinary Coulomb-like repulsion and
  scattering.
- **Molecular state** ($r < \rho$): the particles are bound, oscillating
  between their initial separation $r_0$ and $r = 0$. The sign reversal
  of $\mu_{\text{eff}}$ turns repulsion into effective attraction.

No continuous trajectory can cross $r = \rho$. Weber called this a
"molecular movement" (Sixth Memoir, §9.9).

### The Planetary Atom Model

The **planetary atom** combines a sub-critical nucleus with an orbiting
unlike charge:

| Particle | Charge | Role |
|---|---|---|
| 1 | $+q$ | Nucleus (bound sub-critically with particle 2) |
| 2 | $+q$ | Nucleus (bound sub-critically with particle 1) |
| 3 | $-q$ | Orbiter (Coulomb-attracted to net charge $+2q$) |

The nucleus oscillates at period $T_{\text{nuc}} \approx 2\sqrt{2}\,r_{\text{nuc}}/c$
(fast), while the orbiter has period
$T_{\text{orb}} \approx 2\pi R / v_{\text{circ}}$ (slow). The natural
timescale separation $R \gg r_{\text{nuc}}$ keeps the orbiter far from the
nucleus, preventing three-body instability.

### This Run

This notebook uses the **live streaming** animation viewer, which integrates the system in real time without precomputing the full trajectory. Use the speed slider to control the integration rate.

At $\eta_{\text{orb}} = 0.8$ (80% of circular speed), the orbiter has
more kinetic energy than the $\eta = 0.7$ case and traces a **wide
elliptical orbit** with apoapsis extending to $\sim 2.5R$. This serves
as the baseline for comparison with the Zöllner notebooks, which use
the same $\eta = 0.8$ but add gravitational enhancement.

**Near-circular variant**: setting `eta = 0.7` in cell 5 gives the
near-circular orbit (η = 0.7, apoapsis ~1.1R). Change one line and re-run.

**References**: Weber, Sixth Memoir (1871) §§9.8--9.17; Frauenfelder &
Weber, *Anal. Math. Phys.* **14**:31 (2024); see
`research/theory/CriticalRadiusAndLikeChargeAttraction.md`,
`research/theory/InitialConditions.md`, and
`research/investigations/ThreeBodyBoundStates.md`.

In [ ]:
using WeberElectrodynamics
using LinearAlgebra
using Printf
using GLMakie  # or CairoMakie, WGLMakie

## 1. System Construction and Physical Parameters

In [ ]:
# Physical parameters
m = 1.0               # equal masses
q_pos = 1.0           # positive charges (nucleus)
q_neg = -1.0          # negative charge (orbiter)
c = 4.0

# Derived quantities
mu_nuc = m * m / (m + m)          # nucleus reduced mass = 0.5
rho = q_pos^2 / (mu_nuc * c^2)   # critical radius = 0.125
M_nuc = 2m                         # nucleus total mass
mu_orb = M_nuc * m / (M_nuc + m)  # orbiter reduced mass = 2/3

# Nucleus parameters
r_nuc = 0.05   # sub-critical nucleus separation
T_nuc = 2 * sqrt(2) * r_nuc / c   # nucleus oscillation period

# Orbiter parameters
R = 1.0        # orbiter distance from nucleus COM
Q_eff = 2.0    # effective charge seen by orbiter (q1 + q2)
v_circ = sqrt(Q_eff / (mu_orb * R))  # circular orbit speed
T_orb = 2 * pi * R / v_circ          # orbiter orbital period

# Integration parameters
dt = 1e-4
bounce_r = 0.02

system = HamiltonianSystem(3, 2)

@printf("Three-body planetary atom:\n")
@printf("  Particles:  %d (2D)\n", system.n_particles)
@printf("  DOF:        %d\n", system.degrees_of_freedom)
@printf("\nNucleus (particles 1, 2):\n")
@printf("  q1 = q2 = +%.1f, m = %.1f\n", q_pos, m)
@printf("  r_nuc = %.4f < rho = %.4f\n", r_nuc, rho)
@printf("  T_nuc ~ %.6f\n", T_nuc)
@printf("\nOrbiter (particle 3):\n")
@printf("  q3 = %.1f, m3 = %.1f\n", q_neg, m)
@printf("  R = %.2f, mu_orb = %.4f\n", R, mu_orb)
@printf("  v_circ = %.4f, T_orb ~ %.4f\n", v_circ, T_orb)
@printf("  T_orb / T_nuc = %.1f  (timescale separation)\n", T_orb / T_nuc)
@printf("\nIntegration:\n")
@printf("  dt = %.0e, bounce_r = %.2f, c = %.1f\n", dt, bounce_r, c)

## 2. Initial Conditions and Problem Setup

In [ ]:
function make_planetary_atom_ic(r_nuc, R, eta_orb, m, q_pos, q_neg, c)
    x1 = -r_nuc / 2;  y1 = 0.0
    x2 = +r_nuc / 2;  y2 = 0.0
    x3 = 0.0;          y3 = R

    M_nuc = 2m
    mu_orb = M_nuc * m / (M_nuc + m)
    Q_eff = 2 * abs(q_pos * q_neg)
    v_circ = sqrt(Q_eff / (mu_orb * R))
    v_orb = eta_orb * v_circ

    px3 = m * v_orb;  py3 = 0.0
    px1 = -px3 / 2;   py1 = 0.0
    px2 = -px3 / 2;   py2 = 0.0

    M_total = 3m
    cx = (m * x1 + m * x2 + m * x3) / M_total
    cy = (m * y1 + m * y2 + m * y3) / M_total

    q0 = [x1 - cx, y1 - cy, x2 - cx, y2 - cy, x3 - cx, y3 - cy]
    p0 = [px1, py1, px2, py2, px3, py3]
    return q0, p0, v_circ, v_orb
end

eta = 0.8
tmax = Inf

q0, p0, vc, vorb = make_planetary_atom_ic(r_nuc, R, eta, m, q_pos, q_neg, c)

prob = HamiltonianProblem(system, (0.0, tmax), q0, p0;
    masses = [m, m, m], charges = [q_pos, q_pos, q_neg], c = c, dt = dt,
    zollner=ZollnerOptions())

@printf("Planetary atom: eta = %.1f, v_orb = %.4f (v_circ = %.4f)\n", eta, vorb, vc)

## 3. Live Animation

The streaming animation viewer integrates the system in real time,
displaying rolling trajectories, energy, momentum, angular momentum,
and phase space. Use the **Speed** slider to control how many integration
steps are computed per frame.

In [ ]:
animate_weber(prob; buffer_size = 2000, tail_length = 200, compute_batch = 1)